# 02 — SfM-30k data, DINOv2 smoke checks, pooling pilot, and feature cache

Run this notebook before a full cache. It validates token layout and preprocessing, runs the SfM-only pooling-temperature pilot, then calls the resumable cache script after a temperature is locked.

In [ ]:
%pip install -q -e .
from pathlib import Path
import torch

from cbir.backbone import FrozenDinoV2Extractor
from cbir.config import load_project_config
from cbir.data.sfm import Sfm30kMetadata, SfmImageDirectoryReader, SfmMatImageReader
from cbir.features import FeatureExtractionRunner
from cbir.plotting import SeriesData, plot_series

CONFIG_PATH = Path('configs/extraction_sfm30k.yaml')
cfg = load_project_config(CONFIG_PATH)
print(cfg)

## Phase 0: square/rectangular token smoke test

This must show a 16×16 patch grid for 224×224 and a 16×10 grid for 224×140. The official intermediate API should return genuine patches only; the adapter asserts the register-token convention.

In [ ]:
extractor = FrozenDinoV2Extractor(cfg.backbone)
for hw in [(224, 224), (224, 140)]:
    x = torch.randn(1, 3, *hw)
    outputs = extractor.extract_intermediate_tokens(x, (3, 7, 11))
    print(hw, [(item.block_index, item.patch_grid_hw, tuple(item.patches.shape)) for item in outputs])

## Metadata validation and pooling-temperature pilot

First run scripts/prepare_sfm30k.py in Colab to fetch metadata and the chosen image source. Select a deterministic 1–2k image sample spanning both splits. The pilot computes all candidate temperatures from the same backbone forwards; do not create the full cache until a value is documented and copied into the extraction configuration.

In [ ]:
metadata = Sfm30kMetadata.from_official_files(cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
print(metadata.validate())
records = list(metadata.records('train'))[:1000]

if cfg.sfm.image_root is not None:
    image_reader = SfmImageDirectoryReader(cfg.sfm.image_root)
else:
    image_reader = SfmMatImageReader(cfg.sfm.image_mat_path)
try:
    images = [
        (record.image_id, image_reader.read(record.image_id) if cfg.sfm.image_root is not None else image_reader.read(record.split, int(record.image_locator)))
        for record in records
    ]
finally:
    if isinstance(image_reader, SfmMatImageReader):
        image_reader.close()

runner = FeatureExtractionRunner(extractor, cfg.preprocess, cfg.pooling)
pilot = runner.pilot_pooling_temperatures(
    images,
    temperatures=(0.05, 0.1, 0.2, 0.5, 1.0, 2.0),
    backbone_batch_size=32,
)
series = {
    str(tau): SeriesData(x=list(range(len(pilot.layer_indices))), y=values.mean(dim=0).tolist())
    for tau, values in pilot.entropy_by_temperature.items()
}
plot_series(series, title='Mean normalized pooling entropy', xlabel='Layer position', ylabel='Entropy');

## Full resumable cache

After recording the chosen pooling temperature in the YAML config, first use the limit-500 command to test cache/Drive recovery. Then omit the limit option. The script checks local cache first, then Drive, and only computes unresolved features.

In [ ]:
# !python scripts/extract_features.py --config configs/extraction_sfm30k.yaml --limit 500
# !python scripts/extract_features.py --config configs/extraction_sfm30k.yaml --backbone-batch-size 32